<a href="https://colab.research.google.com/github/MonikaBarget/DigitalHistory/blob/master/BibTex_conversion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Code to convert BibTex bibliographic information into APA7, MLA or Chicago Manual citations**

The Python code below makes use of [```pybtex```](https://docs.pybtex.org/api/parsing.html). One of the key operations of the Pybtex API is parsing BibTeX files, which allows for a structured transformation of bibliographic information into other formats. BibTex files are used as a standard format on many publisher websites and in academic library catalogues, but students and researchers often need citations as plaintext in a particular style, or in HTML and MD format for web publishing.

To transform BibTex into valid APA7 citations, this script uses the [pybtex-apa7-style plugin](https://github.com/cproctor/pybtex-apa7-style?tab=readme-ov-file#readme). Please note that not all special citation cases can be covered by this plugin, but the developers are interested in your feedback. Consider opening an issue on GitHub and submit your sample BibTex files if APA7 output was not generated correctly.

**Note:** Other citation styles such as MLA and Chicago will be added in the future but are not active yet! They have, therefore, been hidden from the style dropdown you will see once running the code.

In [ ]:
# Install required libraries
!pip install pybtex ipywidgets pybtex-apa7-style

# Import libraries

from pybtex.database.input import bibtex as bibtex_reader
import pybtex.plugin
from pybtex.plugin import find_plugin
from pybtex.backends import plaintext
from pybtex.database import parse_string
from IPython.display import display, Markdown
import ipywidgets as widgets
from ipywidgets import interact, interactive_output

In [ ]:
def bibtex_to_citation(bibfile, style):
    cit_style = find_plugin('pybtex.style.formatting', style)()
    HTML = find_plugin('pybtex.backends', 'html')()
    MD = find_plugin('pybtex.backends', 'markdown')()
    TXT = find_plugin('pybtex.backends', 'text')()
    # Parse BibTex file
    try:
      formatted_bib = cit_style.format_bibliography(parse_string(bibfile['content'].decode('utf-8'), 'bibtex'))
    except Exception as e:
      return f"Error when parsing BibTex: {str(e)}."
    try:
      bibinfo_md = " ".join(entry.text.render(MD) for entry in formatted_bib)
      bibinfo_html = "<br>".join(entry.text.render(HTML) for entry in formatted_bib)
      bibinfo_txt = " ".join(entry.text.render(TXT) for entry in formatted_bib)
      return [bibinfo_md, bibinfo_html, bibinfo_txt]
    except Exception as e:
      return f"Error when reformatting citation: {str(e)}."

    # Select style formatter depending on user input
    formatter = None
    if style.lower() == "apa7":
        formatter = pybtex.plugin.find_plugin("pybtex.style.formatting", "apa")()
    elif style.lower() == "mla":
        formatter = pybtex.plugin.find_plugin("pybtex.style.formatting", "mla")()
    elif style.lower() == "chicago":
        formatter = pybtex.plugin.find_plugin("pybtex.style.formatting", "chicago")()
    else:
        return "Error: Invalid style. Use 'apa', 'mla', or 'chicago'."

# Create interactive widgets
file_upload = widgets.FileUpload(
    accept='.bib',  # Accepted file extension
    multiple=False,  # Allow only one file upload at a time!
    description='Upload BibTeX File (.bib)',
)

style_dropdown = widgets.Dropdown(
    options=['---', 'apa7'], # MLA and Chicago have been removed because they are not working yet!
    value='---',
    description='Style:',
    disabled=False,
)

format_dropdown = widgets.Dropdown(
    options=['---', 'HTML', 'plaintext', 'Markdown'],
    value='---',
    description='Format:',
    disabled=False,
)

# Display the output
output = widgets.Output()

def on_button_click(b):
    try:
        bibfile = next(iter(file_upload.value.values()))
        print(f"Using content from file: {bibfile['metadata']['name']}")
    except Exception:
        print("No file uploaded. Please upload a file and try again.")
        return

    style = style_dropdown.value
    format = format_dropdown.value
    cit_list = bibtex_to_citation(bibfile, style)
    if format == "HTML":
      print(f"Formatted citation in HTML ({style.upper()}):\n{cit_list[1]}")
    elif format == "Markdown":
      print(f"Formatted citation MD ({style.upper()}):\n{cit_list[0]}")
    elif format == "plaintext":
      print(f"Formatted citation plaintext ({style.upper()}):\n{cit_list[2]}")
    else:
      print("No format selected. Try again!")
      return

button = widgets.Button(description="Generate Citation")
button.on_click(on_button_click)

# Display the UI for interactive input
# Arrange widgets
ui = widgets.VBox([
    file_upload,
    widgets.HBox([style_dropdown, format_dropdown]),
    button,
    output
])
display(ui)

 [Pybtex-apa7-style](https://github.com/cproctor/pybtex-apa7-style?tab=readme-ov-file#readmeis) a fork of naeka's [pybtex-apa-style](https://github.com/naeka/pybtex-apa-style), which targeted APA6. The APA7 version is maintained by [Chris Proctor](https://github.com/cproctor), Assistant Professor of Learning Sciences at the University of Buffalo, NY. The script provided here is an adaptation by Monika Barget, Maastricht University, for research and teaching.